In [1]:
# Import libraries
import pandas as pd

# Load data
df = pd.read_excel("../data/raw/OnlineRetail.xlsx")

print(df.shape)
df.head()

(541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [2]:
# Remove missing CustomerID
df = df.dropna(subset=['CustomerID'])

# Remove returns (negative quantity)
df = df[df['Quantity'] > 0]

# Remove zero or negative prices
df = df[df['UnitPrice'] > 0]

# Convert date
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Create revenue column
df['Revenue'] = df['Quantity'] * df['UnitPrice']

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


In [3]:
# Create month column
df['Month'] = df['InvoiceDate'].dt.to_period('M')

# Aggregate
agg_df = df.groupby(['StockCode', 'Month']).agg({
    'Quantity': 'sum',
    'UnitPrice': 'mean',
    'Revenue': 'sum'
}).reset_index()

agg_df.rename(columns={
    'Quantity': 'UnitsSold',
    'UnitPrice': 'AvgPrice'
}, inplace=True)

agg_df.head()

,StockCode,Month,UnitsSold,AvgPrice,Revenue
0,10002,2010-12,224,0.85,190.40
1,10002,2011-01,337,0.85,286.45
2,10002,2011-02,50,0.85,42.50
3,10002,2011-03,23,0.85,19.55
4,10002,2011-04,189,0.85,160.65


In [4]:
# Save agg_df as agg_data
agg_df.to_csv("../data/processed/agg_data.csv", index=False)